In [1]:
# 1. Gỡ bỏ sạch sẽ các thư viện default của Colab có khả năng gây xung đột
!pip uninstall -y transformers sentence-transformers torch torchvision torchaudio

# 2. Cài đặt Torch 2.4.0 (chuẩn cu121) và Transformers 4.39.3 (tương thích Mamba)
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.39.3 packaging triton==3.0.0

# 3. Tải và tự động đổi tên file wheel của Causal-Conv1d
!wget -qO causal_conv1d-1.4.0-cp312-cp312-linux_x86_64.whl "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0%2Bcu122torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

# 4. Tải và tự động đổi tên file wheel của Mamba-SSM
!wget -qO mamba_ssm-2.2.4-cp312-cp312-linux_x86_64.whl "https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

# 5. Cài đặt trực tiếp (tốc độ cao, không tốn disk, không cần build)
!pip install causal_conv1d-1.4.0-cp312-cp312-linux_x86_64.whl
!pip install mamba_ssm-2.2.4-cp312-cp312-linux_x86_64.whl
!pip install yfinance gluonts tqdm utilsforecast pyyaml pandas numpy submitit torchmetrics gpytorch

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 94.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.3 MB/s eta 0:00:00
     ━━

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ngngsonan/SC-Mamba.git /content/SC-Mamba 2>/dev/null || echo 'Repo already cloned'
!cd /content/SC-Mamba && git pull


Mounted at /content/drive
Already up to date.


In [9]:
!git pull

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 4.64 KiB | 365.00 KiB/s, done.
From https://github.com/ngngsonan/SC-Mamba
   cd51d39..f3d0446  main       -> origin/main
Updating cd51d39..f3d0446
Fast-forward
 benchmark/02_test_zeroshot_multi.py           |  65 +++++++++
 core/config.b_mamba2.yaml                     | 142 --------------------
 core/config.v_multivariate_exchange_rate.yaml | 127 ------------------
 core/mamba4cast_large_5e5_args.yaml           |   1 -
 core/models.py                                |   7 +-
 insights/Insights_Certainty.ipynb             | 112 ----------------
 insights/Insights_Explainability.ipynb        | 130 ------------------
 insights/__init__.py                          |   0
 logs/log_error.txt                            | 186 ++++++++++++++++----------
 9 files cha

# Train N=1

## config


In [4]:
import os

# Detect project root based on environment
if os.path.exists('/content/SC-Mamba/core/train.py'):
    PROJECT_ROOT = '/content/SC-Mamba'           # Colab
elif os.path.exists(os.path.join(os.getcwd(), 'core', 'train.py')):
    PROJECT_ROOT = os.getcwd()                   # Already at project root
else:
    # Notebook inside benchmark/ → go up one level
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)

print(f'📁 PROJECT_ROOT = {PROJECT_ROOT}')
print(f'   CWD          = {os.getcwd()}')

# Sanity checks
checks = [
    ('core/train.py',                     'Training script'),
    ('core/eval_real_dataset.py',          'Evaluation script'),
    ('core/real_data_args.yaml',           'Eval config'),
    ('data/data_provider/data_loader.py',  'Data loader'),
    ('data/scripts/store_real_datasets.py','Dataset generator'),
]
all_ok = True
for path, label in checks:
    exists = os.path.exists(path)
    status = '✅' if exists else '❌'
    print(f'   {status} {label:30s} → {path}')
    if not exists: all_ok = False

assert all_ok, f'❌ Some paths not found! CWD={os.getcwd()} is wrong.'
print('\n✅ All paths verified. Notebook is ready.')

📁 PROJECT_ROOT = /content/SC-Mamba
   CWD          = /content/SC-Mamba
   ✅ Training script                → core/train.py
   ✅ Evaluation script              → core/eval_real_dataset.py
   ✅ Eval config                    → core/real_data_args.yaml
   ✅ Data loader                    → data/data_provider/data_loader.py
   ✅ Dataset generator              → data/scripts/store_real_datasets.py

✅ All paths verified. Notebook is ready.


In [5]:
# =====================================================================
# DATASET REGISTRY — All 17 Mamba4Cast benchmark datasets
# Columns: (gluonts_key, pred_len, freq, display_name, domain)
# Source: Table 3 of Mamba4Cast (arxiv:2410.09385)
# =====================================================================
DATASET_CONFIGS = [
    ('car_parts_without_missing',      12, 'M', 'Car Parts',        'Retail'),
    ('cif_2016',                       12, 'M', 'CIF 2016',         'Finance/Competition'),
    ('covid_deaths',                   30, 'D', 'Covid Deaths',     'Epidemiology'),
    ('ercot',                          24, 'H', 'ERCOT Load',       'Energy'),
    ('exchange_rate',                  30, 'B', 'Exchange Rate',    'Forex Finance'),
    ('fred_md',                        12, 'M', 'FRED-MD',          'Macroeconomics'),
    ('hospital',                       12, 'M', 'Hospital',         'Healthcare'),
    ('m1_monthly',                     18, 'M', 'M1 Monthly',       'Economics'),
    ('m1_quarterly',                    8, 'Q', 'M1 Quarterly',     'Economics'),
    ('m3_monthly',                     18, 'M', 'M3 Monthly',       'Economics'),
    ('m3_quarterly',                    8, 'Q', 'M3 Quarterly',     'Economics'),
    ('nn5_daily_without_missing',      56, 'D', 'NN5 Daily',        'Banking'),
    ('nn5_weekly',                      8, 'W', 'NN5 Weekly',       'Banking'),
    ('tourism_monthly',                24, 'M', 'Tourism Monthly',  'Tourism'),
    ('tourism_quarterly',               8, 'Q', 'Tourism Quarterly','Tourism'),
    ('traffic',                        24, 'H', 'Traffic',          'Transportation'),
    ('weather',                        30, 'D', 'Weather',          'Meteorology'),
]

# =====================================================================
# ⚙️  USER CONFIGURATION — Edit below
# =====================================================================
# SELECTED_DATASETS options:
#   'all'                        → all 17 datasets (~60-90 min on T4)
#   ['cif_2016', 'm1_monthly']   → specific subset
# Quick validation (~10 min): set QUICK_VALIDATE = True
SELECTED_DATASETS = 'all'
MODEL_VER = 'v_config06'
MODEL_NAME        = f'SCMamba_{MODEL_VER}'    # MUST match generate_model_save_name(config) = f'SCMamba_{version}'
QUICK_VALIDATE    = False            # True = 6 fast datasets only

QUICK_KEYS = ['cif_2016', 'm1_quarterly', 'nn5_weekly', 'tourism_quarterly', 'm1_monthly', 'hospital']

if QUICK_VALIDATE:
    selected = [d for d in DATASET_CONFIGS if d[0] in QUICK_KEYS]
elif SELECTED_DATASETS == 'all':
    selected = DATASET_CONFIGS
else:
    selected = [d for d in DATASET_CONFIGS if d[0] in SELECTED_DATASETS]

print(f'📋 Selected {len(selected)} datasets:')
for key, pred, freq, name, domain in selected:
    print(f'   [{domain:20s}] {name:25s} freq={freq}, pred_len={pred}')


📋 Selected 17 datasets:
   [Retail              ] Car Parts                 freq=M, pred_len=12
   [Finance/Competition ] CIF 2016                  freq=M, pred_len=12
   [Epidemiology        ] Covid Deaths              freq=D, pred_len=30
   [Energy              ] ERCOT Load                freq=H, pred_len=24
   [Forex Finance       ] Exchange Rate             freq=B, pred_len=30
   [Macroeconomics      ] FRED-MD                   freq=M, pred_len=12
   [Healthcare          ] Hospital                  freq=M, pred_len=12
   [Economics           ] M1 Monthly                freq=M, pred_len=18
   [Economics           ] M1 Quarterly              freq=Q, pred_len=8
   [Economics           ] M3 Monthly                freq=M, pred_len=18
   [Economics           ] M3 Quarterly              freq=Q, pred_len=8
   [Banking             ] NN5 Daily                 freq=D, pred_len=56
   [Banking             ] NN5 Weekly                freq=W, pred_len=8
   [Tourism             ] Tourism Monthly  

In [6]:
import yaml, os

config = {
    'seed': 42,
    'debugging': False,
    'scaler': 'min_max',
    'sin_pos_enc': False,
    'sin_pos_const': False,
    'sub_day': True,            # ← BẬT: sub-daily encoding (quan trọng cho ERCOT/Traffic hourly)
    'encoding_dropout': 0.1,
    'handle_constants_model': True,
    'num_assets': 1,            # Channel-Independent: mỗi series là 1 asset độc lập

    'ssm_config': {
        'bidirectional': False,
        'enc_conv': True,
        'init_dil_conv': True,
        'enc_conv_kernel': 5,
        'init_conv_kernel': 5,
        'init_conv_max_dilation': 3,
        'global_residual': False,
        'in_proj_norm': False,
        'initial_gelu_flag': True,
        'linear_seq': 15,
        'mamba2': True,         # ← BẬT: Mamba SSM v2 (khớp với Mamba4Cast paper)
        'norm': True,
        'norm_type': 'layernorm',
        'num_encoder_layers': 2,
        'd_state': 128,
        'residual': False,
        'token_embed_len': 1024,
        'block_expansion': 2,
        'chunk_size': 256,        # Mamba2 SSD kernel chunk size; encode_temporal pads seq_len to ≥2×chunk_size (nchunks≥2)
        'headdim': 128           # CRITICAL: must be >= max(BLOCK_SIZE_M)=128; headdim=64 (default) → partial Triton blocks → LLVM crash

    },

    # ---- Optimization (Config B — ~40× nhiều hơn baseline) ----
    'lr_scheduler': 'cosine', #'cosine_warm_restarts',  # Warm restart giúp thoát local minima
    'initial_lr': 5e-5,
    'learning_rate': 1e-7,
    'num_epochs': 50, #       # ← 5× nhiều hơn (was 10)
    'training_rounds': 500,   #600,    # ← 8× nhiều hơn (was 250) batches/epoch
    'validation_rounds': 200,  #200,
    'real_test_interval': 5,    # Chạy real-data eval mỗi 5 epoch
    'real_test_datasets': [d[0] for d in selected[:2]]+['exchange_rate','fred_md'],  # Monitor 2 datasets khi train
    't_max': -1,               # T_0 for CosineAnnealingWarmRestarts (warmup cycle length)

    # ---- SC-Mamba-specific loss (NLL ELBO thay cho MSE của Mamba4Cast) ----
    'loss': 'nll',              # NLL + KL divergence (probabilistic model)
    'beta_kl': 0.1, #0.1,             # Weight của spectral KL term trong ELBO
    'multipoint': True,         # Predict toàn bộ horizon cùng lúc
    'sample_multi_pred': 0.5,   # 50% batches dùng multipoint prediction
    'diag_prints': True,        # ← diagnostic epoch summaries (LR, beta, NLL, KL, grad_norm)

    # ---- Checkpoint ----
    'wandb': False,
    'continue_training': True,  # ⚠️ False vì architecture thay đổi (mamba2=True)
    'model_prefix': '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints',
    'version': MODEL_VER,       # → tên checkpoint: SCMamba_{MODEL_VER}

    # ---- Sequence sampling (khớp với Mamba4Cast: context [30, 256], pred [10, 60]) ----
    'context_len': 256,
    'min_seq_len': 30,
    'max_seq_len': 256,         # ← Tăng từ 128 lên 256 (context window dài hơn)
    'pred_len': 60,             # Max prediction length
    'pred_len_min': 10,         # Min prediction length
    'pred_len_sample': True,    # Sample ngẫu nhiên pred_len → zero-shot generalization tốt hơn
    'batch_size': 64,
    'no_pos_enc': False,        # Dùng positional encoding

    # ---- Synthetic Prior (70% GP + 30% ForecastPFN) ----
    'prior_config': {
        'prior_mix_frac': 0.7,      # 70% dữ liệu từ GP prior
        'curriculum_learning': False,
        'mixup_prob': 0.0,
        'mixup_series': 4,
        'damp_and_spike': True,     # Thêm damping/spike noise để robust hơn
        'damping_noise_ratio': 0.05,
        'spike_noise_ratio': 0.05,
        'spike_signal_ratio': 0.05,
        'spike_batch_ratio': 0.05,
        'fp_options': {
            'linear_random_walk_frac': 0,
            'seasonal_only': False,
            'scale_noise': [0.6, 0.3],  # [low, moderate] noise levels
            'trend_exp': False,
            'harmonic_scale_ratio': 0.4,
            'harmonic_rate': 0.75,
            'trend_additional': True,
            'transition_ratio': 0.0,
        },
        'gp_prior_config': {
            'max_kernels': 6,
            'likelihood_noise_level': 0.4,
            'noise_level': 'random',
            'use_original_gp': False,
            'gaussians_periodic': True,
            'peak_spike_ratio': 0.1,
            'subfreq_ratio': 0.2,
            'periods_per_freq': 0.5,
            'gaussian_sampling_ratio': 0.2,
            'kernel_periods': [4, 5, 7, 21, 24, 30, 60, 120],
            'max_period_ratio': 1.0,
            'kernel_bank': {
                'matern_kernel': 1.5,
                'linear_kernel': 1,
                'periodic_kernel': 5,
                'polynomial_kernel': 0,
                'spectral_mixture_kernel': 0,
            },
        },
    },
}

os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)
config_name = f'config.{MODEL_VER}.yaml'
config_path = os.path.join(PROJECT_ROOT, 'core', config_name)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'✅ Config B written → {config_path}')
print(f'   mamba2={config["ssm_config"]["mamba2"]} | sub_day={config["sub_day"]} | loss={config["loss"]}')
print(f'   epochs={config["num_epochs"]} | rounds={config["training_rounds"]} | batch={config["batch_size"]}')
print(f'   seq=[{config["min_seq_len"]}, {config["max_seq_len"]}] | pred=[{config["pred_len_min"]}, {config["pred_len"]}]')
print(f'   Live monitoring: {config["real_test_datasets"]}')

os.environ['SC_MAMBA_DIAG'] = '1'

✅ Config B written → /content/SC-Mamba/core/config.v_config06.yaml
   mamba2=True | sub_day=True | loss=nll
   epochs=50 | rounds=500 | batch=64
   seq=[30, 256] | pred=[10, 60]
   Live monitoring: ['car_parts_without_missing', 'cif_2016', 'exchange_rate', 'fred_md']


In [7]:
# @title Real validation datasets
import os, subprocess

# =====================================================================
# 1. Verify real validation datasets exist (pkl files)
# =====================================================================
REAL_VAL_DIR = os.path.join(PROJECT_ROOT, 'data', 'real_val_datasets')
os.makedirs(REAL_VAL_DIR, exist_ok=True)

# Check which datasets are needed for real_test_datasets
real_test_ds = config.get('real_test_datasets', [])
padded = 'pad' if False else 'nopad'  # matches real_data_args.yaml pad=false
MAX_LEN = 512

missing = []
for ds in real_test_ds:
    pkl_name = f'{ds}_{padded}_{MAX_LEN}.pkl'
    pkl_path = os.path.join(REAL_VAL_DIR, pkl_name)
    if os.path.exists(pkl_path):
        size_mb = os.path.getsize(pkl_path) / (1024*1024)
        print(f'  ✅ {ds:30s} → {pkl_name} ({size_mb:.1f} MB)')
    else:
        missing.append(ds)
        print(f'  ❌ {ds:30s} → MISSING')

if missing:
    print(f'\n🔄 Generating {len(missing)} missing dataset(s) from GluonTS...')
    print('   This downloads from AWS and converts to pkl. May take 1-5 min per dataset.')
    !python data/scripts/store_real_datasets.py
    # Verify again
    for ds in missing:
        pkl_path = os.path.join(REAL_VAL_DIR, f'{ds}_{padded}_{MAX_LEN}.pkl')
        if os.path.exists(pkl_path):
            print(f'  ✅ {ds} → OK')
        else:
            print(f'  ⚠️  {ds} → STILL MISSING. Check store_real_datasets.py output above.')
else:
    print(f'\n✅ All {len(real_test_ds)} real validation datasets found.')

# =====================================================================
# 2. Resume from checkpoint (if training was interrupted)
# =====================================================================
# Set continue_training=True to resume from the last saved checkpoint.
# The training loop will load optimizer state, scheduler state, and epoch.
RESUME_TRAINING = False    # <---- SET TO True TO RESUME

if RESUME_TRAINING:
    config['continue_training'] = True
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    print('🔄 Resume mode enabled: will load from last checkpoint.')
else:
    print('ℹ️  Fresh training. Set RESUME_TRAINING=True to resume from checkpoint.')

  ✅ car_parts_without_missing      → car_parts_without_missing_nopad_512.pkl (2.0 MB)
  ❌ cif_2016                       → MISSING
  ❌ exchange_rate                  → MISSING
  ✅ fred_md                        → fred_md_nopad_512.pkl (0.8 MB)

🔄 Generating 2 missing dataset(s) from GluonTS...
   This downloads from AWS and converts to pkl. May take 1-5 min per dataset.
  📥 Downloading M3C.xls from GitHub mirror...
  ✅ Saved to /root/.gluonts/datasets/M3C.xls (1716 KB)
  📥 Downloading M1C.xls from GitHub mirror...
  ⚠️  Failed to download M1C.xls: HTTP Error 404: Not Found
      Manual fix: download from https://raw.githubusercontent.com/Mcompetitions/M1-methods/master/M1C.xls
      and place at: /root/.gluonts/datasets/M1C.xls
Dataset nn5_daily_without_missing already exists. Skipping...
Dataset nn5_weekly already exists. Skipping...
Dataset covid_deaths already exists. Skipping...
Dataset weather already exists. Skipping...
Dataset hospital already exists. Skipping...
Dataset fred_md

## Training

In [ ]:
!python core/train.py -c core/config.v_config06.yaml

config:
{'batch_size': 64,
 'beta_kl': 0.1,
 'context_len': 256,
 'continue_training': True,
 'debugging': False,
 'diag_prints': True,
 'encoding_dropout': 0.1,
 'handle_constants_model': True,
 'initial_lr': 5e-05,
 'learning_rate': 1e-07,
 'loss': 'nll',
 'lr_scheduler': 'cosine',
 'max_seq_len': 256,
 'min_seq_len': 30,
 'model_prefix': '/content/drive/MyDrive/Colab '
                 'Notebooks/SCMamba/sc_mamba_checkpoints',
 'multipoint': True,
 'no_pos_enc': False,
 'num_assets': 1,
 'num_epochs': 50,
 'pred_len': 60,
 'pred_len_min': 10,
 'pred_len_sample': True,
 'prior_config': {'curriculum_learning': False,
                  'damp_and_spike': True,
                  'damping_noise_ratio': 0.05,
                  'fp_options': {'harmonic_rate': 0.75,
                                 'harmonic_scale_ratio': 0.4,
                                 'linear_random_walk_frac': 0,
                                 'scale_noise': [0.6, 0.3],
                                 'seasonal_o

In [ ]:
# --- Verify checkpoint saved ---
!ls -lh models/

total 0


## Eval

In [ ]:
!ls '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'

'Bản sao của SCMamba_v_config02_test_best.pth'
'Copy of SCMamba_v_config02_test_best.pth'
'Copy of SCMamba_v_config05_test_best.pth'
 SCMamba_v_config06_best_mase.pth
 SCMamba_v_config06_best.pth
 SCMamba_v_config06_Final.pth
 SCMamba_v_config06.pth
 SCMamba_v_config06_test_best_mase.pth
 SCMamba_v_config06_test_best.pth
 SCMamba_v_config06_test_Final.pth
 SCMamba_v_config06_test.pth


In [ ]:
# !rm -rf '/content/SC-Mamba/data/real_data_evals/SCMamba_v_config06_test_best_mase'
# !rm -rf /root/.gluonts/datasets/nn5_weekly/


In [ ]:
import subprocess, yaml, os
import pandas as pd
from pathlib import Path

# =====================================================================
# SCMamba Evaluation Aggregation Script
# Tự động quét cache, chạy eval_real_dataset.py cho các dataset còn thiếu
# và tổng hợp kết quả (MASE, CRPS, NLL...) ra file CSV.
# =====================================================================

# --- Cấu hình Môi trường ---
os.environ["TRITON_F32_DEFAULT"] = "ieee"
PROJECT_ROOT = '/content/SC-Mamba' # Đổi lại nếu anh chạy local
CONFIG_NAME = 'config.v_config06.yaml' # File YAML chứa tham số train (để extract num_assets và sub_day)
CHECKPOINT_PATH = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints/SCMamba_v_config06_best_mase.pth'

QUICK_VALIDATE = False
SELECTED_DATASETS = 'all'      # Hoặc truyền list: ['cif_2016', 'nn5_weekly']
PRED_STYLE = 'multipoint'      # SC-Mamba luôn dùng multipoint

# Dành cho Quick Validation
QUICK_KEYS = ['cif_2016', 'm1_quarterly', 'nn5_weekly', 'tourism_quarterly', 'm1_monthly', 'hospital']

# (Giả định DATASET_CONFIGS đã được định nghĩa ở cell trước)
# Nếu chưa, hệ thống sẽ báo lỗi. Đảm bảo anh đã run khối chứa định nghĩa 17 dataset.
try:
    DATASET_CONFIGS
except NameError:
    print("⚠️ BẠN CHƯA ĐỊNH NGHĨA DATASET_CONFIGS. Chạy lại cell khởi tạo biến trước khi chạy cell này.")

# --- Thiết lập Đường dẫn & Tên Cache ---
# Lấy chính xác tên checkpoint làm tên thư mục Cache (Giữ nguyên cả chữ _best_mase)
EVAL_MODEL_NAME = os.path.basename(CHECKPOINT_PATH).replace('.pth', '')
EVAL_DIR = Path(PROJECT_ROOT) / 'data' / 'real_data_evals' / EVAL_MODEL_NAME / PRED_STYLE
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Lọc dataset cần evaluate
if QUICK_VALIDATE:
    selected = [d for d in DATASET_CONFIGS if d[0] in QUICK_KEYS]
elif SELECTED_DATASETS == 'all':
    selected = DATASET_CONFIGS
else:
    selected = [d for d in DATASET_CONFIGS if d[0] in SELECTED_DATASETS]

# --- Kiểm tra Cache hiện tại ---
cached, missing = [], []
for gluonts_key, pred_len, freq, name, domain in selected:
    # 512 là MAX_LENGTH cố định chặn trên contextual string trong source
    yml_path = EVAL_DIR / f'{gluonts_key}_512.yml'
    if yml_path.exists():
        cached.append(name)
    else:
        missing.append(name)

print(f"{'='*60}")
print(f"📊 BẮT ĐẦU ĐÁNH GIÁ MÔ HÌNH")
print(f"{'='*60}")
print(f'Model name : {EVAL_MODEL_NAME}')
print(f'Checkpoint : {CHECKPOINT_PATH}')
print(f'Config YAML: {CONFIG_NAME}')
print(f'Eval dir   : {EVAL_DIR}')
print(f'Trạng thái : ✅ Cached: {len(cached)} | ❌ Missing: {len(missing)}')

# --- Gọi Subprocess chạy Evaluation cho dataset chưa có ---
if missing:
    print(f'▶ Sẽ chạy Evaluation cho {len(missing)} dataset mới: {missing}\n')
    cmd = [
        'python', f'{PROJECT_ROOT}/core/eval_real_dataset.py',
        '-c', CHECKPOINT_PATH,
        '-o', EVAL_MODEL_NAME,  # Ép thư mục output bằng chính tên Checkpoint
        '-cfg', os.path.join(PROJECT_ROOT, 'core', CONFIG_NAME) # Để tự lấy tag sub_day=True
    ]

    print(f'💻 Executing: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f'\n⚠️ LỖI: eval_real_dataset.py bị crash (Mã lỗi {result.returncode})')
        print(f'--- STDERR ---\n{result.stderr}')
    else:
        print(f'\n✅ Evaluation Script chạy xong không gặp lỗi.')
else:
    print('\n✅ Tất cả datasets yêu cầu đều đã có Cache. Bỏ qua bước chạy mô hình.')

# --- Đọc Load Kết quả vào Pandas DataFrame ---
results = []
for gluonts_key, pred_len, freq, name, domain in selected:
    yml_path = EVAL_DIR / f'{gluonts_key}_512.yml'

    if not yml_path.exists():
        results.append({'dataset': name, 'key': gluonts_key, 'domain': domain,
                        'freq': freq, 'pred_len': pred_len, 'status': 'MISSING', 'MASE': None})
        continue

    with open(yml_path) as f:
        raw = yaml.safe_load(f)

    if not raw:
        results.append({'dataset': name, 'key': gluonts_key, 'domain': domain,
                        'freq': freq, 'pred_len': pred_len, 'status': 'EMPTY', 'MASE': None})
        continue

    metrics = next(iter(raw.values()), {})
    if metrics.get('mase') is None:
        results.append({'dataset': name, 'key': gluonts_key, 'domain': domain,
                        'freq': freq, 'pred_len': pred_len, 'status': 'PLACEHOLDER', 'MASE': None})
        continue

    results.append({
        'dataset': name, 'key': gluonts_key, 'domain': domain,
        'freq': freq, 'pred_len': pred_len, 'status': 'OK',
        'MASE': metrics.get('mase'), 'MAE': metrics.get('mae'),
        'RMSE': metrics.get('rmse'), 'sMAPE': metrics.get('smape'),
        'NLL': metrics.get('nll'), 'CRPS': metrics.get('crps'),
    })
    print(f'✅ [{domain[:15]:15s}] {name[:20]:20s} MASE={metrics.get("mase","?"):.4f}  CRPS={metrics.get("crps","?"):.4f}')

# Lập bảng tổng kết
df = pd.DataFrame(results)
csv_name = f'{EVAL_MODEL_NAME}_quick_results.csv' if QUICK_VALIDATE else f'{EVAL_MODEL_NAME}_all_results.csv'
csv_path = os.path.join(PROJECT_ROOT, 'results', csv_name)
os.makedirs(os.path.join(PROJECT_ROOT, 'results'), exist_ok=True)
df.to_csv(csv_path, index=False)

# --- In Tổng kết ---
print(f'\n{"="*60}')
print(f'📊 TỔNG KẾT — {EVAL_MODEL_NAME}')
print(f'{"="*60}')

ok = df[df['status'] == 'OK']
if not ok.empty:
    cols = ['dataset', 'domain', 'freq', 'pred_len']
    for m in ['MASE', 'sMAPE', 'CRPS', 'NLL']:
        if m in ok.columns: cols.append(m)

    # Render table (Display trong Jupyter Notebook)
    try:
        display(ok[cols].round(4))
    except NameError:
        print(ok[cols].round(4))

    print(f'\n💎 Avg MASE: {ok["MASE"].mean():.4f}')
    if 'CRPS' in ok.columns:
        print(f'💎 Avg CRPS: {ok["CRPS"].mean():.4f}')

err = df[df['status'] != 'OK']
if not err.empty:
    print(f'\n⚠️ Phát hiện {len(err)} datasets LỖI (Chưa eval được):')
    for _, r in err.iterrows():
        print(f'   [{r["status"]:12s}] {r["dataset"]}')

print(f'\n💾 Đã lưu full bảng biểu tại: {csv_path}')


📊 BẮT ĐẦU ĐÁNH GIÁ MÔ HÌNH
Model name : SCMamba_v_config06_best_mase
Checkpoint : /content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints/SCMamba_v_config06_best_mase.pth
Config YAML: config.v_config06.yaml
Eval dir   : /content/SC-Mamba/data/real_data_evals/SCMamba_v_config06_best_mase/multipoint
Trạng thái : ✅ Cached: 0 | ❌ Missing: 17
▶ Sẽ chạy Evaluation cho 17 dataset mới: ['Car Parts', 'CIF 2016', 'Covid Deaths', 'ERCOT Load', 'Exchange Rate', 'FRED-MD', 'Hospital', 'M1 Monthly', 'M1 Quarterly', 'M3 Monthly', 'M3 Quarterly', 'NN5 Daily', 'NN5 Weekly', 'Tourism Monthly', 'Tourism Quarterly', 'Traffic', 'Weather']

💻 Executing: python /content/SC-Mamba/core/eval_real_dataset.py -c /content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints/SCMamba_v_config06_best_mase.pth -o SCMamba_v_config06_best_mase -cfg /content/SC-Mamba/core/config.v_config06.yaml

✅ Evaluation Script chạy xong không gặp lỗi.
✅ [Retail         ] Car Parts            MASE=1.1611  CRPS=0.494

,dataset,domain,freq,pred_len,MASE,sMAPE,CRPS,NLL
0,Car Parts,Retail,M,12,1.1611,0.8818,4.946000e-01,13698.0996
1,CIF 2016,Finance/Competition,M,12,1.3252,0.0917,1.291092e+06,8.0640
2,Covid Deaths,Epidemiology,D,30,11.9262,0.2773,4.045016e+02,524317.3750
3,ERCOT Load,Energy,H,24,4.1283,0.0843,8.584585e+02,8.1720
4,Exchange Rate,Forex Finance,B,30,1.7127,0.0062,8.400000e-03,-3.2075
5,FRED-MD,Macroeconomics,M,12,2.0001,0.0762,3.784492e+03,5.2454
6,Hospital,Healthcare,M,12,0.8270,0.0933,1.759480e+01,4.0958
7,M1 Monthly,Economics,M,18,1.3756,0.0956,1.682082e+03,5.9276
8,M1 Quarterly,Economics,Q,8,2.1106,0.0966,1.779003e+03,5.8954
9,M3 Monthly,Economics,M,18,1.0524,0.0792,5.451987e+02,8.1059



💎 Avg MASE: 2.4450
💎 Avg CRPS: 77436.6604

💾 Đã lưu full bảng biểu tại: /content/SC-Mamba/results/SCMamba_v_config06_best_mase_all_results.csv


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd, os

results_path = os.path.join(PROJECT_ROOT, 'results', f'{MODEL_NAME}_all_results.csv')
results_path = '/content/SC-Mamba/results/SCMamba_v_len128_round250_all_results.csv'
if not os.path.exists(results_path):
    print('⚠️  Run Step 4 first.')
else:
    df = pd.read_csv(results_path)
    df_ok = df[df['status'] == 'OK'].copy()

    metric_cols = [m for m in ['MASE', 'sMAPE', 'CRPS'] if m in df_ok.columns]
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(7*len(metric_cols), 7))
    if len(metric_cols) == 1: axes = [axes]

    fig.suptitle(f'SC-Mamba ({MODEL_NAME}) — Zero-Shot Benchmark', fontsize=14, fontweight='bold')

    for ax, metric in zip(axes, metric_cols):
        data = df_ok.set_index('dataset')[metric].sort_values()
        med = data.median()
        colors = ['#43A047' if v <= med else '#EF5350' for v in data]
        bars = ax.barh(data.index, data.values, color=colors, edgecolor='white')
        ax.axvline(med, color='#212121', linestyle='--', alpha=0.6, linewidth=1.2, label=f'Median={med:.3f}')
        ax.set_xlabel(metric, fontsize=11)
        ax.set_title(f'{metric} by Dataset (↓ better)', fontsize=12)
        ax.legend(fontsize=9)
        for bar, val in zip(bars, data.values):
            ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                   f'{val:.3f}', va='center', fontsize=8)

    plt.tight_layout()
    out_path = f'results/{MODEL_NAME}_benchmark_summary.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved → {out_path}')

⚠️  Run Step 4 first.


# Training exchange_rate num_assets > 1

In [10]:
# @title training
# ============================================================
# 🚀 SC-Mamba — Cross-Asset Causal Graph Training
# Trigger: num_assets > 1 + real_train_datasets → auto-activate
#          MultivariateRealDataset (train from scratch, no finetune)
# ============================================================
import yaml, os, subprocess
from pathlib import Path

PROJECT_ROOT   = '/content/SC-Mamba'
CHECKPOINT_DIR = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'

# ── Danh sách các thí nghiệm Multivariate ───────────────────
EXPERIMENTS = {
    'exchange_rate': {
        'num_assets'        : 8,     # Exchange Rate: đúng 8 đồng tiền
        'pred_len'          : 20,
        'prior_mix_frac'    : 0.5,   # 50% synthetic GP + 50% real aligned
        'batch_size'        : 8,     # B×N = 8×8 = 64 series/step (T4-safe)
        'beta_kl'           : 0.2,   # (Giảm từ 0.5 xuống 0.2 để tránh Gradient Vanishing cho tau)
        'beta_anneal_epochs': 10,    # (Kéo dài warmup)
        'training_rounds'   : 200,
        'sub_day'           : False, # Daily freq → no hour/minute features
    },
}

# ── Template cấu hình (shared across experiments) ───────────
BASE_CONFIG = {
    # Core settings
    'seed'                 : 42,
    'debugging'            : False,
    'scaler'               : 'min_max',
    'sin_pos_enc'          : False,
    'sin_pos_const'        : False,
    'encoding_dropout'     : 0.1,
    'handle_constants_model': True,

    # Backbone architecture (Mamba2 — matches existing checkpoints)
    'ssm_config': {
        'bidirectional'         : False,
        'enc_conv'              : True,
        'init_dil_conv'         : True,
        'enc_conv_kernel'       : 5,
        'init_conv_kernel'      : 5,
        'init_conv_max_dilation': 3,
        'global_residual'       : False,
        'in_proj_norm'          : False,
        'initial_gelu_flag'     : True,
        'linear_seq'            : 15,
        'mamba2'                : True,
        'norm'                  : True,
        'norm_type'             : 'layernorm',
        'num_encoder_layers'    : 2,
        'd_state'               : 128,
        'residual'              : False,
        'token_embed_len'       : 1024,
        'block_expansion'       : 2,
        'chunk_size'            : 256,
        'headdim'               : 128,
    },

    # Training schedule
    'num_epochs'        : 30, #30,   # Train from scratch — epoch 0
    'validation_rounds' : 50,
    'real_test_interval': 5,

    # Learning rate
    'lr_scheduler' : 'cosine',
    'initial_lr'   : 5e-5,
    'learning_rate': 1e-5,      # (Tăng từ 1e-7 lên 1e-5 để kích thích tau học lại)
    't_max'        : -1,        # Auto-set to num_epochs

    # Context/prediction length
    'context_len'      : 256,
    'min_seq_len'      : 60,
    'max_seq_len'      : 256,
    'pred_len_sample'  : False, # Fixed pred_len (stable Cross-Asset windows)
    'pred_len_min'     : 10,

    # Loss
    'loss'             : 'nll',
    'multipoint'       : True,
    'sample_multi_pred': 0.3,
    'diag_prints'      : True,

    # Checkpoint
    'model_prefix'    : CHECKPOINT_DIR,
    'wandb'           : False,
    'continue_training': True, # ← Train từ epoch 0 (không load checkpoint cũ)

    # Synthetic prior settings (shared structure, mix_frac set per experiment)
    'prior_config': {
        'curriculum_learning'  : False,
        'mixup_prob'           : 0.0,
        'mixup_series'         : 4,
        'damp_and_spike'       : False,
        'damping_noise_ratio'  : 0.0,
        'spike_noise_ratio'    : 0.0,
        'spike_signal_ratio'   : 0.0,
        'spike_batch_ratio'    : 0.0,
        'fp_options': {
            'linear_random_walk_frac': 0,
            'seasonal_only'  : False,
            'scale_noise'    : [0.6, 0.3],
            'trend_exp'      : False,
            'harmonic_scale_ratio': 0.4,
            'harmonic_rate'  : 0.75,
            'trend_additional': True,
            'transition_ratio': 0.0,
        },
        'gp_prior_config': {
            'max_kernels'            : 6,
            'likelihood_noise_level' : 0.4,
            'noise_level'            : 'random',
            'use_original_gp'        : False,
            'gaussians_periodic'     : True,
            'peak_spike_ratio'       : 0.1,
            'subfreq_ratio'          : 0.2,
            'periods_per_freq'       : 0.5,
            'gaussian_sampling_ratio': 0.2,
            'kernel_periods'         : [4, 5, 7, 21, 24, 30, 60, 120],
            'max_period_ratio'       : 1.0,
            'kernel_bank': {
                'matern_kernel'         : 1.5,
                'linear_kernel'         : 1,
                'periodic_kernel'       : 5,
                'polynomial_kernel'     : 0,
                'spectral_mixture_kernel': 0,
            },
        },
    },
}

# ── Run experiments ──────────────────────────────────────────
for dataset_name, exp_cfg in EXPERIMENTS.items():
    print(f"\n{'='*65}")
    print(f"🚀 CROSS-ASSET TRAINING: {dataset_name.upper()}")
    print(f"   num_assets    : {exp_cfg['num_assets']}")
    print(f"   prior_mix_frac: {exp_cfg['prior_mix_frac']}  "
          f"(synthetic {exp_cfg['prior_mix_frac']*100:.0f}% + "
          f"real {(1-exp_cfg['prior_mix_frac'])*100:.0f}%)")
    print(f"   num_epochs    : {BASE_CONFIG['num_epochs']}  (train from scratch)")
    print(f"{'='*65}")

    # Build final config by merging BASE + experiment-specific keys
    config = {**BASE_CONFIG}
    config['num_assets']          = exp_cfg['num_assets']
    config['sub_day']             = exp_cfg['sub_day']
    config['pred_len']            = exp_cfg['pred_len']
    config['batch_size']          = exp_cfg['batch_size']
    config['beta_kl']             = exp_cfg['beta_kl']
    config['beta_anneal_epochs']  = exp_cfg['beta_anneal_epochs']
    config['training_rounds']     = exp_cfg['training_rounds']
    config['version']             = f'v_multi_{dataset_name}'

    # real_train_datasets + real_test_datasets — both point to same dataset
    config['real_train_datasets'] = [dataset_name]  # ← activates MultivariateRealDataset
    config['real_test_datasets']  = [dataset_name]

    # Inject prior_mix_frac into prior_config
    config['prior_config'] = {
        **BASE_CONFIG['prior_config'],
        'prior_mix_frac': exp_cfg['prior_mix_frac'],
    }

    # Write config to file
    config_path = f'{PROJECT_ROOT}/core/config.{config["version"]}.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    print(f"📝 Config saved: {config_path}")

    # Launch training — stream logs live
    cmd = ['python', f'{PROJECT_ROOT}/core/train.py', '-c', config_path]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.stdout.close()
    rc = proc.wait()

    if rc == 0:
        print(f"\n✅ {dataset_name} training complete.")
    else:
        print(f"\n❌ {dataset_name} training failed (exit code {rc}). Stopping.")
        break



🚀 CROSS-ASSET TRAINING: EXCHANGE_RATE
   num_assets    : 8
   prior_mix_frac: 0.5  (synthetic 50% + real 50%)
   num_epochs    : 30  (train from scratch)
📝 Config saved: /content/SC-Mamba/core/config.v_multi_exchange_rate.yaml
config:
{'batch_size': 8,
 'beta_anneal_epochs': 10,
 'beta_kl': 0.2,
 'context_len': 256,
 'continue_training': True,
 'debugging': False,
 'diag_prints': True,
 'encoding_dropout': 0.1,
 'handle_constants_model': True,
 'initial_lr': 5e-05,
 'learning_rate': 1e-05,
 'loss': 'nll',
 'lr_scheduler': 'cosine',
 'max_seq_len': 256,
 'min_seq_len': 60,
 'model_prefix': '/content/drive/MyDrive/Colab '
                 'Notebooks/SCMamba/sc_mamba_checkpoints',
 'multipoint': True,
 'num_assets': 8,
 'num_epochs': 30,
 'pred_len': 20,
 'pred_len_min': 10,
 'pred_len_sample': False,
 'prior_config': {'curriculum_learning': False,
                  'damp_and_spike': False,
                  'damping_noise_ratio': 0.0,
                  'fp_options': {'harmonic_rate': 0.

In [11]:
!ls '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'

SCMamba_v_config06_best_mase.pth
SCMamba_v_config06_best.pth
SCMamba_v_config06_Final.pth
SCMamba_v_config06.pth
SCMamba_v_multi_exchange_rate_best_mase.pth
SCMamba_v_multi_exchange_rate_best.pth
SCMamba_v_multi_exchange_rate_Final.pth
SCMamba_v_multi_exchange_rate.pth
SCMamba_v_multivariate_cif_2016_best_mase.pth
SCMamba_v_multivariate_cif_2016_best.pth
SCMamba_v_multivariate_cif_2016_Final.pth
SCMamba_v_multivariate_cif_2016.pth
SCMamba_v_multivariate_exchange_rate_best_mase.pth
SCMamba_v_multivariate_exchange_rate_best.pth
SCMamba_v_multivariate_exchange_rate_Final.pth
SCMamba_v_multivariate_exchange_rate.pth
SCMamba_v_multivariate_fred_md_best_mase.pth
SCMamba_v_multivariate_fred_md_best.pth
SCMamba_v_multivariate_fred_md.pth


In [12]:
# @title 01_Ckp_Eval_CrossAsset
"""
01_Ckp_Eval_CrossAsset.py
==============================
Ablation evaluation script: num_assets=1 (univariate) vs num_assets=8 (cross-asset graph).

Run directly via terminal:
python benchmark/01_Ckp_Eval_CrossAsset.py
"""

import sys
import os
import torch
import numpy as np
import pandas as pd
import yaml
from pprint import pprint

# Setup paths and environment
# sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
os.environ['TRITON_F32_DEFAULT'] = 'ieee'  # Triton compiler workaround

from core.models import SCMamba_Forecaster
from core.real_data_val_pipeline import validate_on_real_dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# CKPT_DIR = os.path.join(os.path.dirname(__file__), '..', 'sc_mamba_checkpoints')
SCALER   = 'min_max'   # must match training config

# If you run on Colab, you might need to override CKPT_DIR:
CKPT_DIR = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'

# SSM config — must match your training config exactly
SSM_CONFIG = {
    'mamba2'             : True,
    'num_encoder_layers' : 2,
    'd_state'            : 128,
    'headdim'            : 128,
    'block_expansion'    : 2,
    'token_embed_len'    : 1024,
    'chunk_size'         : 256,
    'linear_seq'         : 15,
    'norm'               : True,
    'norm_type'          : 'layernorm',
    'residual'           : False,
    'global_residual'    : False,
    'bidirectional'      : False,
    'in_proj_norm'       : False,
    'enc_conv'           : True,
    'enc_conv_kernel'    : 5,
    'init_dil_conv'      : True,
    'init_conv_kernel'   : 5,
    'init_conv_max_dilation': 3,
    'initial_gelu_flag'  : True,
}

print('Setup complete. DEVICE =', DEVICE)


def load_model(ckpt_path: str, num_assets: int, ssm_config: dict) -> SCMamba_Forecaster:
    """Load SCMamba_Forecaster from checkpoint."""
    model = SCMamba_Forecaster(N_assets=num_assets, ssm_config=ssm_config).to(DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    # Support both raw state_dict and wrapped {'model_state_dict': ...}
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state, strict=True)
    model.eval()
    print(f'  ✅ Loaded: {os.path.basename(ckpt_path)} | N_assets={num_assets}')
    return model


def evaluate_model(
    model,
    dataset: str,
    scaler: str = 'min_max',
    sub_day: bool = False,
    label: str = 'model',
) -> dict:
    """
    Run validate_on_real_dataset and return a dict of metrics.
    Prints a formatted row suitable for comparison table.
    """
    print(f'\n🔍 Evaluating [{label}] on {dataset} ...')
    mase_, mae_, rmse_, smape_, nll_, crps_ = validate_on_real_dataset(
        dataset, model, DEVICE, scaler, subday=sub_day
    )
    result = {
        'label'  : label,
        'dataset': dataset,
        'MASE'   : round(mase_,  4),
        'MAE'    : round(mae_,   4),
        'RMSE'   : round(rmse_,  4),
        'SMAPE'  : round(smape_, 4),
        'NLL'    : round(nll_,   4),   # ↓ better  (probabilistic, unique to SC-Mamba)
        'CRPS'   : round(crps_,  4),   # ↓ better
    }
    print(f'  MASE={mase_:.4f} | MAE={mae_:.4f} | RMSE={rmse_:.4f} | '
          f'SMAPE={smape_:.4f} | NLL={nll_:.4f} | CRPS={crps_:.4f}')
    return result


def main():
    # Checkpoint naming convention (set by train.py › generate_model_save_name):
    #   N=1  → SCMamba_<version_univariate>_best.pth   (trained with num_assets=1)
    #   N=8  → SCMamba_v_multivariate_exchange_rate_best.pth (trained with num_assets=8)

    # Adjust the paths below to match your environment.
    # Example using Colab paths:
    ckpt_dir = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'

    ABLATION_CONFIGS = [
        {
            'label'      : 'N=1 (Univariate)',
            'ckpt'       : f'{ckpt_dir}/SCMamba_v_config06_best_mase.pth',
            'num_assets' : 1,
            'dataset'    : 'exchange_rate',
            'sub_day'    : False,
        },
        {
            'label'      : 'N=8 (Cross-Asset Graph)',
            'ckpt'       : f'{ckpt_dir}/SCMamba_v_multi_exchange_rate_best_mase.pth',
            'num_assets' : 8,
            'dataset'    : 'exchange_rate',
            'sub_day'    : False,
        },
        {
            'label'      : 'N=8 (Cross-Asset Graph)',
            'ckpt'       : f'{ckpt_dir}/SCMamba_v_multi_exchange_rate_best.pth',
            'num_assets' : 8,
            'dataset'    : 'exchange_rate',
            'sub_day'    : False,
        },
    ]

    results = []
    for cfg in ABLATION_CONFIGS:
        print(f'\n━━━━ {cfg["label"]} ━━━━')
        if not os.path.exists(cfg['ckpt']):
            print(f"⚠️ Warning: Checkpoint not found -> {cfg['ckpt']}")
            print("Skipping this configuration.\n")
            continue

        model = load_model(cfg['ckpt'], cfg['num_assets'], SSM_CONFIG)
        row   = evaluate_model(
            model,
            dataset = cfg['dataset'],
            scaler  = SCALER,
            sub_day = cfg['sub_day'],
            label   = cfg['label'],
        )
        results.append(row)

        # Free memory between runs
        del model
        torch.cuda.empty_cache()

    if not results:
        print("No valid results found to display.")
        return

    # ── Results Table + Interpretation ────────────────────────────────────────
    df = pd.DataFrame(results).set_index('label')
    print('\n' + '='*70)
    print('  ABLATION RESULTS: exchange_rate — N=1 vs N=8')
    print('='*70)
    print(df[['MASE','MAE','RMSE','SMAPE','NLL','CRPS']].to_string())
    print('='*70)
    print('  ↓ lower is better for all metrics')
    print()

    # Relative improvement (N=8 vs N=1)
    if len(df) == 2:
        base  = df.iloc[0]   # N=1
        multi = df.iloc[1]   # N=8
        print('  Relative delta (N=8 - N=1) / |N=1|:')
        for m in ['MASE','MAE','RMSE','SMAPE','NLL','CRPS']:
            delta_pct = (multi[m] - base[m]) / (abs(base[m]) + 1e-10) * 100
            arrow = '🟢' if delta_pct < 0 else '🔴'
            print(f'    {arrow}  {m:6s}: {delta_pct:+.1f}%')
        print()

# if __name__ == '__main__':
main()


Setup complete. DEVICE = cuda

━━━━ N=1 (Univariate) ━━━━
  ✅ Loaded: SCMamba_v_config06_best_mase.pth | N_assets=1

🔍 Evaluating [N=1 (Univariate)] on exchange_rate ...
reading data from /content/SC-Mamba/data/real_val_datasets/exchange_rate_nopad_512.pkl
test 40
5
  [DIAG:pad] seq_len=542 → padded=768 (pad=226, waste=29.4%), chunk_size=256
  [DIAG:spectral] N_assets=1, freq_bins=1, mask=[0.1225, 0.9262], alpha=0.99, tau=1.9973, KL=0.052015, fft_identity_diff=0.00e+00
  [DIAG:pad] seq_len=542 → padded=768 (pad=226, waste=29.4%), chunk_size=256
  [DIAG:spectral] N_assets=1, freq_bins=1, mask=[0.1225, 0.9288], alpha=0.99, tau=1.9973, KL=0.053165, fft_identity_diff=0.00e+00
  [DIAG:pad] seq_len=542 → padded=768 (pad=226, waste=29.4%), chunk_size=256
  [DIAG:spectral] N_assets=1, freq_bins=1, mask=[0.1225, 0.9343], alpha=0.99, tau=1.9973, KL=0.055880, fft_identity_diff=0.00e+00
  MASE=4.8074 | MAE=0.0219 | RMSE=0.0226 | SMAPE=0.0148 | NLL=-2.4862 | CRPS=0.0150

━━━━ N=8 (Cross-Asset Graph